# Layout algorithm lab

This notebook loads complete saved layouts from `data/training/`, uses each file's `wysokosc_rolki` as the authoritative generated result, compares other algorithms on the same inputs, and shows result tables, SVG previews, and comparison plots.

In [ ]:
from pathlib import Path
import sys
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import HTML, SVG, display

ROOT = Path.cwd().resolve()
while not (ROOT / "optimizer").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("No 'optimizer' directory found in the current path or any parent directories.")
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.notebooks import (
    area_decreasing, benchmark, first_fit_decreasing, max_rects,
    genetic_baseline,
    load_json, load_training_layout, random_search,
)

In [ ]:
layout_paths = sorted((ROOT / 'data/training').glob('*.json'))
if not layout_paths:
    raise ValueError('No saved layouts found in data/training/.')
saved_cases = []
for layout_path in layout_paths:
    layout = load_json(layout_path)
    machine_width, items = load_training_layout(layout_path)
    saved_cases.append({
        'case_id': layout_path.stem,
        'layout_path': layout_path,
        'layout': layout,
        'machine_width': machine_width,
        'items': items,
    })
print(f'Saved layouts loaded: {len(saved_cases)}')

Saved layouts loaded: 440


In [ ]:
# Create a fresh persistent table containing the saved genetic benchmark only.
RESULTS_TABLE_PATH = ROOT / "experiments/layout_algorithm_results.csv"

benchmark_records = {
    record["case_id"]: record
    for record in load_json(ROOT / "experiments/benchmark_results.json")
}

benchmark_rows = []
for case in saved_cases:
    layout = case["layout"]
    rectangles = layout.get("elementy", [])
    saved_height = float(layout["wysokosc_rolki"])
    used_area = sum(
        float(item.get("w", 0)) * float(item.get("h", 0))
        for item in rectangles
    )
    benchmark_record = benchmark_records.get(case["case_id"], {})
    benchmark_rows.append({
        "case_id": case["case_id"],
        "algorithm": "Saved genetic layout",
        "height_mm": saved_height,
        "used_area_mm2": used_area,
        "utilization_percent": (
            used_area / (case["machine_width"] * saved_height) * 100
            if saved_height else 0.0
        ),
        "placed": len(rectangles),
        "missing": 0,
        "seconds": benchmark_record.get("baseline_seconds", np.nan),
        "machine_width": case["machine_width"],
        "layout_path": str(case["layout_path"]),
    })

results_df = pd.DataFrame(benchmark_rows)
height_categories = pd.qcut(
    results_df.set_index("case_id")["height_mm"].rank(method="first"),
    q=3,
    labels=["Short", "Medium", "Tall"],
)
results_df["genetic_height_category"] = results_df["case_id"].map(height_categories)
results_df.to_csv(RESULTS_TABLE_PATH, index=False)


In [ ]:
def add_algorithm_results(algorithm, algorithm_name, **benchmark_options):
    """Run one algorithm, replace its previous rows, and save the newest table."""
    global results_df

    rows = []
    for case in saved_cases:
        result = benchmark(
            case["items"],
            case["machine_width"],
            algorithms={algorithm_name: algorithm},
            **benchmark_options,
        )[0]
        rows.append({
            "case_id": case["case_id"],
            "algorithm": algorithm_name,
            "height_mm": result["height_mm"],
            "used_area_mm2": result["used_area_mm2"],
            "utilization_percent": result["utilization_percent"],
            "placed": result["placed"],
            "missing": result["missing"],
            "seconds": result["seconds"],
            "machine_width": case["machine_width"],
            "layout_path": str(case["layout_path"]),
            "genetic_height_category": results_df.loc[
                results_df["case_id"] == case["case_id"],
                "genetic_height_category",
            ].iloc[0],
        })

    new_rows = pd.DataFrame(rows)
    results_df = pd.concat(
        [
            results_df[results_df["algorithm"] != algorithm_name],
            new_rows,
        ],
        ignore_index=True,
    )
    results_df.to_csv(RESULTS_TABLE_PATH, index=False)
    return results_df


In [ ]:
def run_algorithm_sequential(algorithm, algorithm_name, legacy_algorithm=genetic_baseline, **benchmark_options):
    """Run the new algorithm and legacy genetic baseline case by case."""
    ordered_cases = sorted(
        saved_cases,
        key=lambda case: (case["layout_path"].name.startswith("gen_"), case["layout_path"].name.lower()),
    )
    new_results = []
    new_total_started = time.perf_counter()
    legacy_total_elapsed = 0.0
    for index, case in enumerate(ordered_cases, start=1):
        print(f"[{index}/{len(ordered_cases)}] {case['layout_path'].name}")
        population_size = benchmark_options.get("population_size", 40)
        generations = benchmark_options.get("generations", 80)
        configured_algorithm = lambda items, width: algorithm(
            items,
            width,
            population_size=population_size,
            generations=generations,
        )
        configured_legacy_algorithm = lambda items, width: legacy_algorithm(
            items,
            width,
            population_size=population_size,
            generations=generations,
        )
        new_started = time.perf_counter()
        new_result = benchmark(
            case["items"],
            case["machine_width"],
            algorithms={algorithm_name: configured_algorithm},
        )[0]
        new_elapsed = time.perf_counter() - new_started
        print(
            f"    {algorithm_name}: height={new_result['height_mm']:.2f} mm, "
            f"placed={new_result['placed']}, missing={new_result['missing']}, "
            f"elapsed={new_elapsed:.3f}s"
        )
        new_results.append(new_result)

        legacy_started = time.perf_counter()
        legacy_result = benchmark(
            case["items"],
            case["machine_width"],
            algorithms={"Legacy Genetic Algorithm": configured_legacy_algorithm},
        )[0]
        legacy_elapsed = time.perf_counter() - legacy_started
        legacy_total_elapsed += legacy_elapsed
        print(
            f"    Legacy Genetic Algorithm: height={legacy_result['height_mm']:.2f} mm, "
            f"placed={legacy_result['placed']}, missing={legacy_result['missing']}, "
            f"elapsed={legacy_elapsed:.3f}s"
        )
    print(f"Total {algorithm_name} time: {time.perf_counter() - new_total_started:.3f}s")
    print(f"Total Legacy Genetic Algorithm time: {legacy_total_elapsed:.3f}s")
    return new_results


In [ ]:
category_summary_df = (
    results_df
    .groupby(
        ["genetic_height_category", "algorithm"],
        observed=True,
        as_index=False,
    )
    .agg(
        cases=("case_id", "nunique"),
        mean_time_seconds=("seconds", "mean"),
        mean_height_mm=("height_mm", "mean"),
        mean_utilization_percent=("utilization_percent", "mean"),
        mean_placed=("placed", "mean"),
        total_missing=("missing", "sum"),
    )
    .sort_values(["genetic_height_category", "mean_height_mm"])
)

display(
    category_summary_df.style.format({
        "mean_time_seconds": "{:.3f}",
        "mean_height_mm": "{:.2f}",
        "mean_utilization_percent": "{:.2f}",
        "mean_placed": "{:.2f}",
    })
)
